# Game Bot Creator -- Imitation-Learning ("Record & Play") Policy Training

Trains a small behavioral-cloning policy on a human demonstration recorded by Game Bot Creator's `imitation_recorder.py` prototype: given the current screen frame, predict which keys/mouse buttons should be held and where the mouse should be. See `docs/imitation-learning-action-investigation.md` in the main game-bot repo for the full design discussion -- this notebook is the first concrete artifact of that investigation's training half.

**Model: MobileNetV3-Small, single-frame, stateless** -- deliberately the cheapest, least risky architecture choice (see the investigation doc's "architecture choice affects export difficulty" note), picked to prove the whole record -> train -> export -> run pipeline shape before attempting a temporal (LSTM) variant, which needs a materially different, stateful ONNX export shape (hidden state as an explicit input/output, threaded across ticks by the caller) that this notebook deliberately does not attempt yet.

**Two output heads, both from the same backbone features:**
- `action_logits`: multi-label classification over every distinct key/mouse-button the recorded demonstration(s) actually used (the action vocabulary is *discovered from the data itself*, not fixed in advance -- see step 3). Multi-label, not single-label, since more than one key can be legitimately held at once (e.g. forward + strafe).
- `mouse_xy`: normalized (0-1) mouse position, trained only on frames where the demonstration's own recorded position was inside the capture frame (`mouse_in_frame`) -- a frame where the cursor was elsewhere contributes no mouse-position signal, not a wrong one.

**Expected input: a zip of `imitation_recorder.py`'s own dataset-directory output** -- one or more `session_<timestamp>/` folders, each holding `frames/frame_NNNNNN.png` + `actions.jsonl` (one JSON line per frame: `frame`, `t`, `keys` (raw Windows virtual-key codes), `mouse_buttons`, `mouse_x`, `mouse_y`, `mouse_in_frame`). **No app-side packaging/upload/Kaggle-push step exists yet for this action type** (unlike the object-detection notebooks, which `cv_training.py`/the AI tab already wire up end-to-end) -- for now, zip `imitation_recorder.py`'s dataset dir by hand and upload it the same way this notebook's Colab branch expects.

**Known gap carried over from the recorder itself, not fixed here:** `keys`/`mouse_buttons` are raw Windows virtual-key codes / button names, not the string key names `backends.InputBackend.hold_key`/`press_combo` expect (e.g. `"w"`) -- the app-side runtime piece (not yet built) will need a vk-code -> key-name mapping before this model's predictions can actually drive `InputBackend`. Recorded verbatim here since round-tripping that mapping isn't this notebook's job.

**Status: hands-on validated locally (CPU-only, a small synthetic recorded dataset) that this exact pipeline shape -- vocabulary discovery, dataset loading, model, training loop, ONNX export, onnxruntime inference -- runs correctly end-to-end and produces a loadable model with the right output shapes.** Not yet run on Colab/Kaggle itself, and not yet validated against a real, meaningful gameplay demonstration -- treat the way every other notebook in this repo treats its own "not yet verified against a real run" disclaimer: the mechanism is proven, the actual training quality on a real task is not.

## 1. Install dependencies

`torch`/`torchvision` are **not** pinned here -- both Colab and Kaggle ship a working, CUDA-matched torch preinstalled, and pip-installing a different one is a well-known way to break that CUDA binding (the object-detection notebooks make the same choice: they pin `ultralytics`/`onnx`/`onnxruntime`, never `torch` itself, even though `ultralytics` depends on it). `onnx`/`onnxruntime` pins match this repo's other notebooks for consistency.

In [ ]:
!pip install -q "onnx==1.22.0" "onnxruntime==1.29.0"

### Optional: check what hardware this session got

Purely informational, same as the object-detection notebooks' own version of this cell. This model is small enough to train on CPU in a reasonable time for a prototype-sized dataset, but a GPU still helps.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Upload and unpack the dataset

Same Kaggle-vs-Colab auto-detection convention as this repo's other notebooks (`ON_KAGGLE`, detected by whether the expected dataset shape is actually present under `/kaggle/input`, not just whether that directory exists -- see the YOLO notebook's own note on why a fixed-depth glob broke this detection on a real Kaggle run). Expects one or more `session_*/actions.jsonl` files somewhere under the extracted root.

**For anything but a small dataset, upload via Google Drive, not the button below.** A real run with 9 recorded sessions (1066 frames) produced a 970MB zip, and Colab's interactive upload widget (`google.colab.files.upload()`) failed on it outright (`Found 0 session(s)` after upload finished) -- that widget base64-encodes the whole file through the browser-kernel message bridge, the same slow/fragile path this repo's own notebooks already work around on the *download* side (see step 8's own Google Drive alternative) -- it's just as unreliable for a large *upload*, and more recorded sessions/higher-resolution frames only make this worse. Set `USE_GOOGLE_DRIVE_UPLOAD = True` and `DRIVE_ZIP_PATH` below to the dataset zip's location after uploading it to your Drive yourself (drag-and-drop at drive.google.com handles large files far more reliably than this notebook's own upload button ever will).

In [ ]:
import glob
import shutil
import zipfile
from pathlib import Path

RAW_DIR = Path("dataset_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

# Set True (and the path below) after uploading the dataset zip to your own Google Drive
# -- see this cell's markdown for why the upload button is unreliable for anything but a
# small dataset.
USE_GOOGLE_DRIVE_UPLOAD = False
DRIVE_ZIP_PATH = "/content/drive/MyDrive/dataset.zip"

candidates = glob.glob("/kaggle/input/**/actions.jsonl", recursive=True)
ON_KAGGLE = bool(candidates)
if ON_KAGGLE:
    # Recursive glob, not a fixed single-level path -- Kaggle mounts datasets under
    # /kaggle/input/datasets/<owner>/<dataset>/, not a flat /kaggle/input/<dataset>/,
    # same real gotcha the YOLO notebook's own dataset-detection cell already hit.
    # Each match is one session's actions.jsonl -- copy every session folder found.
    RAW_DIR.mkdir(parents=True)
    session_dirs = {Path(c).parent.parent for c in candidates}
    for session_dir in session_dirs:
        shutil.copytree(session_dir, RAW_DIR / session_dir.name)
elif USE_GOOGLE_DRIVE_UPLOAD:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(DRIVE_ZIP_PATH).is_file():
        raise RuntimeError(
            f"DRIVE_ZIP_PATH ({DRIVE_ZIP_PATH}) doesn't exist -- check the path matches "
            "where you actually uploaded the zip in your Drive."
        )
    with zipfile.ZipFile(DRIVE_ZIP_PATH) as zf:
        zf.extractall(RAW_DIR)
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(RAW_DIR)

session_dirs = sorted(p for p in RAW_DIR.glob("**/session_*") if (p / "actions.jsonl").exists())
if not session_dirs:
    # A zip of imitation_recorder.py's dataset dir extracts session_*/ folders directly at
    # its root -- but Colab's files.upload() extraction, or a differently-structured zip,
    # can land them one level deeper. This mirrors the YOLO notebook's own recursive-glob
    # fix for the same class of "where did the files actually land" surprise.
    session_dirs = sorted(p.parent for p in RAW_DIR.glob("**/actions.jsonl"))
print(f"Found {len(session_dirs)} session(s):", [p.name for p in session_dirs])
assert session_dirs, "No session_*/actions.jsonl found -- check the uploaded zip's contents."

## 3. Discover the action vocabulary, load frame/label pairs, and split train/val

The action vocabulary (which keys/mouse buttons this model predicts) is **discovered from the data itself** by scanning every session's `actions.jsonl` for every distinct key code / mouse button that ever appears -- not a fixed list, since different games and different demonstrations use entirely different keys. A 90/10 train/val split, same convention (and same "too small for a real split" fallback) as this repo's object-detection notebooks.

In [ ]:
import json
import random

entries = []  # list of (frame_path, label_dict)
for session_dir in session_dirs:
    for line in (session_dir / "actions.jsonl").read_text(encoding="utf-8").strip().splitlines():
        label = json.loads(line)
        entries.append((session_dir / "frames" / label["frame"], label))
print(f"Loaded {len(entries)} frame/label pairs from {len(session_dirs)} session(s)")

key_vocab = sorted({k for _, label in entries for k in label["keys"]})
button_vocab = sorted({b for _, label in entries for b in label["mouse_buttons"]})
ACTION_VOCAB = [f"key_{k}" for k in key_vocab] + [f"button_{b}" for b in button_vocab]
print("Discovered action vocab:", ACTION_VOCAB)
assert ACTION_VOCAB, (
    "No keys or mouse buttons were ever held in this recording -- nothing for the model "
    "to learn. Record a demonstration that actually presses something."
)

import cv2
sample_frame = cv2.imread(str(entries[0][0]))
FRAME_H, FRAME_W = sample_frame.shape[:2]
print(f"Native frame size: {FRAME_W}x{FRAME_H}")

VALID_FRACTION = 0.1
random.seed(0)
shuffled = entries[:]
random.shuffle(shuffled)
n_valid = max(1, int(len(shuffled) * VALID_FRACTION)) if len(shuffled) > 1 else 0
val_entries, train_entries = shuffled[:n_valid], shuffled[n_valid:]
if not val_entries:
    print("Dataset too small for a real validation split -- reusing the training frames for "
          "'val' instead, same as this repo's object-detection notebooks in the same situation. "
          "Early-stopping/reported val loss won't reflect genuine held-out performance until "
          "there are enough frames for VALID_FRACTION to carve out at least one.")
    val_entries = train_entries
print(f"train: {len(train_entries)}  val: {len(val_entries)}")

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

IMG_SIZE = (224, 224)  # MobileNetV3's native ImageNet-pretrained input size

class DemoDataset(Dataset):
    def __init__(self, entries):
        self.entries = entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        frame_path, label = self.entries[idx]
        img = cv2.imread(str(frame_path))
        img = cv2.resize(img, IMG_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        held = {f"key_{k}" for k in label["keys"]} | {f"button_{b}" for b in label["mouse_buttons"]}
        action_vec = np.array([1.0 if name in held else 0.0 for name in ACTION_VOCAB], dtype=np.float32)

        mouse_valid = 1.0 if label.get("mouse_in_frame") else 0.0
        if mouse_valid:
            mouse_xy = np.array([label["mouse_x"] / FRAME_W, label["mouse_y"] / FRAME_H], dtype=np.float32)
        else:
            mouse_xy = np.zeros(2, dtype=np.float32)

        return (
            torch.from_numpy(img),
            torch.from_numpy(action_vec),
            torch.from_numpy(mouse_xy),
            torch.tensor(mouse_valid, dtype=torch.float32),
        )

BATCH_SIZE = 16
train_loader = DataLoader(DemoDataset(train_entries), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(DemoDataset(val_entries), batch_size=BATCH_SIZE, shuffle=False)

## 4. Model: MobileNetV3-Small backbone, two heads

ImageNet-pretrained backbone (same "don't train a vision backbone from scratch on a tiny custom dataset" reasoning as the object-detection notebooks' own pretrained-checkpoint choice), classifier head replaced with two linear heads sharing the same pooled features -- one multi-label (actions), one 2-unit regression (mouse position).

In [ ]:
import torch.nn as nn
import torchvision.models as tvm

class PolicyNet(nn.Module):
    def __init__(self, n_actions: int):
        super().__init__()
        backbone = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        in_features = backbone.classifier[3].in_features
        backbone.classifier[3] = nn.Identity()  # keep the 1024-d pooled feature, drop ImageNet's own 1000-class head
        self.backbone = backbone
        self.action_head = nn.Linear(in_features, n_actions)
        self.mouse_head = nn.Linear(in_features, 2)

    def forward(self, x):
        feat = self.backbone(x)
        return self.action_head(feat), self.mouse_head(feat)

model = PolicyNet(len(ACTION_VOCAB))
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("Training on:", device)

## 5. Train

Multi-label action loss (`BCEWithLogitsLoss`, since more than one action can be true per frame) plus a mouse-position loss (`MSELoss`) masked to only the frames where the demonstration's own mouse position was inside the frame -- a plain mean over `mouse_valid` would incorrectly pull invalid (zeroed) positions toward the loss otherwise. `PATIENCE`-based early stopping on validation loss, same idea as the object-detection notebooks' own early stopping, with the best-checkpoint weights explicitly restored before export -- not just whatever's left in memory after the loop ends (the same discipline the YOLO notebook's own export step calls out by name).

In [ ]:
import copy

EPOCHS = 60
PATIENCE = 8

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
action_loss_fn = nn.BCEWithLogitsLoss()
mouse_loss_fn = nn.MSELoss(reduction="none")

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, n_batches = 0.0, 0
    for imgs, action_vecs, mouse_xys, mouse_valids in loader:
        imgs, action_vecs = imgs.to(device), action_vecs.to(device)
        mouse_xys, mouse_valids = mouse_xys.to(device), mouse_valids.to(device)
        with torch.set_grad_enabled(train):
            action_logits, mouse_pred = model(imgs)
            action_loss = action_loss_fn(action_logits, action_vecs)
            per_sample_mouse_loss = mouse_loss_fn(mouse_pred, mouse_xys).mean(dim=1)
            valid_count = mouse_valids.sum().clamp(min=1.0)
            mouse_loss = (per_sample_mouse_loss * mouse_valids).sum() / valid_count
            loss = action_loss + mouse_loss
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0
for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    print(f"epoch {epoch + 1}/{EPOCHS}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}"
          f"{' (best)' if improved else ''}")
    if epochs_without_improvement >= PATIENCE:
        print(f"No improvement for {PATIENCE} epochs -- stopping early.")
        break

model.load_state_dict(best_state)  # best checkpoint, not whatever's left after the loop -- see markdown above
model.eval()
print(f"Restored best checkpoint (val_loss={best_val_loss:.4f})")

## 6. Export to ONNX + action vocabulary metadata

Unlike the single-class object-detection notebooks, this model's output only means something alongside the action vocabulary it was trained against -- `action_vocab.json` is exported alongside `policy.onnx` for exactly that reason (no equivalent file needed for object detection's single fixed `"object"` class). `dynamo=False` forces PyTorch's classic TorchScript-based ONNX exporter -- confirmed directly against a real local run that PyTorch's newer default (`torch.export`-based) exporter needs the separate `onnxscript` package not installed here, and the classic exporter is otherwise equivalent for this plain (no control-flow) model.

In [ ]:
model_cpu = model.to("cpu")
dummy = torch.zeros(1, 3, IMG_SIZE[1], IMG_SIZE[0], dtype=torch.float32)
torch.onnx.export(
    model_cpu, dummy, "policy.onnx",
    input_names=["frame"], output_names=["action_logits", "mouse_xy"],
    dynamic_axes=None, opset_version=17, dynamo=False,
)

with open("action_vocab.json", "w", encoding="utf-8") as f:
    json.dump({"action_vocab": ACTION_VOCAB, "img_size": list(IMG_SIZE)}, f, indent=2)

print("Exported policy.onnx + action_vocab.json")

## 7. Sanity-check the export (optional)

Same idea as the object-detection notebooks' own sanity-check step -- not proof the policy is any good, just proof the exported graph runs and produces plausible-shaped output before downloading it.

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession("policy.onnx", providers=["CPUExecutionProvider"])
check_frame, _, _, _ = DemoDataset(val_entries)[0]
action_logits_out, mouse_xy_out = sess.run(None, {"frame": check_frame.unsqueeze(0).numpy()})
print("action_logits shape:", action_logits_out.shape, "expected:", (1, len(ACTION_VOCAB)))
print("mouse_xy shape:", mouse_xy_out.shape, "expected:", (1, 2))

action_probs = 1 / (1 + np.exp(-action_logits_out[0]))
print("Predicted action probabilities:", dict(zip(ACTION_VOCAB, action_probs.round(3))))
print("Predicted (normalized) mouse xy:", mouse_xy_out[0])

## 8. Download the trained model

Both files are needed together -- zipped into one download for convenience, same reasoning as `action_vocab.json`'s own note above.

In [ ]:
with zipfile.ZipFile("policy_export.zip", "w") as zf:
    zf.write("policy.onnx")
    zf.write("action_vocab.json")

if ON_KAGGLE:
    # Kaggle captures every file left in /kaggle/working/ as this kernel's output -- no
    # scripted poll/pull exists for this action type yet (see this notebook's intro), so
    # for now this just needs to be fetched by hand from the kernel's Output tab.
    print("On Kaggle: policy_export.zip left in /kaggle/working/ -- download it from the kernel's Output tab.")
else:
    from google.colab import files
    files.download("policy_export.zip")